# K-nearest neighbours (KNN)
The breast cancer data, choosing k, decision surfaces and the failure cases.

In [ ]:
import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load the data: 569 tumours, 30 numeric measurements, target 0 = malignant, 1 = benign.
# (The original CSV also has an "id" column and an empty "Unnamed: 32" column; scikit-learn's copy has neither.)
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
print(X.shape)
print(y.value_counts())
X.head()

## Split, scale, train, evaluate

In [ ]:
# Hold back 20% of the rows as the test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
print(X_train.shape, X_test.shape)

# Learn the mean and standard deviation from the training set only, then apply them to both sets
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# k = 5 is the default number of neighbours
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_s, y_train)            # "training" only stores the rows
y_pred = knn.predict(X_test_s)
print("accuracy with scaling   ", accuracy_score(y_test, y_pred))

# The same model on unscaled data, for comparison
raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
print("accuracy without scaling", accuracy_score(y_test, raw.predict(X_test)))

## Choosing k
Heuristic: the square root of the number of training rows, made odd.

In [ ]:
print("sqrt(n) =", np.sqrt(len(X_train)))

# Experiment: one model per k, scored on the test set (quick, but it lets the test set pick k)
ks = range(1, 16)
test_acc = [accuracy_score(y_test, KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train).predict(X_test_s)) for k in ks]
go.Figure(go.Scatter(x=list(ks), y=test_acc, mode="lines+markers")).update_layout(
    template="simple_white", xaxis_title="k", yaxis_title="test accuracy")

In [ ]:
# The correct way: choose k with cross-validation on the training set only, then test once.
# The pipeline re-fits the scaler inside every fold, so no fold sees the others' statistics.
model = make_pipeline(StandardScaler(), KNeighborsClassifier())
search = GridSearchCV(model, {"kneighborsclassifier__n_neighbors": list(ks)}, cv=5)
search.fit(X_train, y_train)
print("best k by CV:", search.best_params_, "CV accuracy", round(search.best_score_, 3))
print("final test accuracy:", search.score(X_test, y_test))

## Decision surface on two columns
The same code as `app.py` (run `python app.py` for the k slider).

In [ ]:
from app import figure_for
figure_for(5)

## Failure case 1: prediction is slow on big data

In [ ]:
rng = np.random.default_rng(0)
queries = rng.random((1000, 30))
for n in [10_000, 100_000, 500_000]:
    Xb, yb = rng.random((n, 30)), rng.integers(0, 2, n)
    m = KNeighborsClassifier(algorithm="brute")
    t0 = time.perf_counter(); m.fit(Xb, yb); t1 = time.perf_counter(); m.predict(queries); t2 = time.perf_counter()
    print(f"n = {n:>7,}: fit {1000 * (t1 - t0):6.1f} ms, predict 1000 rows {1000 * (t2 - t1):6.1f} ms")

## Failure case 5: imbalanced classes

In [ ]:
# 98% of rows in class 0, 2% in class 1
Xi, yi = make_classification(n_samples=5000, n_features=10, weights=[0.98], flip_y=0, class_sep=0.5, random_state=0)
a, b, c, d = train_test_split(Xi, yi, test_size=0.3, stratify=yi, random_state=0)
p = KNeighborsClassifier(n_neighbors=5).fit(a, c).predict(b)
print("rare-class rows in test:", d.sum())
print("accuracy", round(accuracy_score(d, p), 3), "| rare class found (recall)", round(recall_score(d, p), 3))